In [35]:
import pandas as pd
import numpy as np
import librosa

In [36]:
train_df = pd.read_pickle('train_embedded.pkl')
test_df = pd.read_pickle("test_embedded.pkl")
train_df.head()

,signal_data,language,speaker,label,gender,c,length,w2v_base_emb_pooled,w2v_base_emb,w2v_xlr_emb,w2v_xlr_emb_pooled,mel_spec,mel_spec_padded,mel_spec_pooled
176,"[0.000152587890625, 0.000823974609375, 0.00082...",SA,A,4,F,c,9501,"[-0.015912058, 0.23642024, 0.12297839, 0.21679...","[[0.031614933, 0.27669638, 0.15805711, 0.19895...","[[-0.33289763, -0.16397952, -0.044866655, 0.08...","[-0.043815523, -0.08047871, 0.052105945, 0.042...","[[[-55.064064, -53.257996, -59.722855, -48.224...","[[[-55.064064, -53.257996, -59.722855, -48.224...","[-50.13206, -48.11771, -48.631584, -45.271175,..."
165,"[-3.0517578125e-05, 0.0, 0.0, 0.0, 3.051757812...",FR,F,2,F,c,6201,"[0.38915816, 0.3792101, 0.090889215, 0.4901088...","[[0.40043145, 0.3911669, 0.12556341, 0.4663496...","[[-0.28900322, -0.17436437, -0.04185667, 0.092...","[-0.04911978, -0.11596217, 0.06338483, 0.03236...","[[[-64.447174, -65.83354, -78.590675, -77.9472...","[[[-64.447174, -65.83354, -78.590675, -77.9472...","[-57.80567, -54.742844, -53.918205, -49.423088..."
126,"[9.1552734375e-05, 0.000152587890625, 0.000244...",FR,B,3,F,c,10001,"[0.17522134, 0.23353295, -0.1157971, 0.3438407...","[[0.4023691, 0.1572093, 0.19292863, 0.04548339...","[[-0.25203943, -0.1384559, -0.065831274, 0.049...","[-0.01860454, -0.07754524, 0.047171906, 0.0199...","[[[-79.21712, -73.76115, -75.72417, -74.95683,...","[[[-79.21712, -73.76115, -75.72417, -74.95683,...","[-66.945656, -60.16159, -57.949112, -54.54152,..."
103,"[0.0, 0.0, 0.0, -0.0078125, 0.0, -0.0078125, -...",SI,C,9,M,c,9001,"[0.1901748, 0.27533138, -0.021832537, 0.112002...","[[0.33527187, 0.33407038, 0.18568435, 0.031803...","[[-0.22652705, -0.19961871, -0.058231324, 0.10...","[-0.0071312133, -0.13924864, 0.030800128, 0.02...","[[[-35.610336, -34.771824, -35.57545, -34.6930...","[[[-35.610336, -34.771824, -35.57545, -34.6930...","[-36.700752, -34.88265, -35.53902, -23.765522,..."
70,"[-0.010894775390625, -0.018310546875, -0.00708...",IT,E,5,F,c,15000,"[0.07980928, 0.08479349, 0.17919528, 0.1372474...","[[0.0787321, 0.28121114, 0.21040522, 0.2661217...","[[-0.25639585, -0.231193, -0.0117138345, 0.057...","[0.050507627, -0.04553041, 0.06576309, 0.01915...","[[[-58.969536, -49.85008, -48.191605, -46.3154...","[[[-58.969536, -49.85008, -48.191605, -46.3154...","[-49.63411, -49.047318, -48.6203, -53.586643, ..."


In [37]:
def get_mel_spectrogram(signal, sr=16000, n_mels=128):
    # 1. Convert list to numpy array and ensure float32
    if isinstance(signal, list):
        signal = np.array(signal, dtype=np.float32)
        
    # 2. Extract the Mel Spectrogram
    # n_mels sets the number of frequency bands
    mel_spec = librosa.feature.melspectrogram(
        y=signal, 
        sr=sr, 
        n_mels=n_mels,
        n_fft=512,    # Window size for the Fourier transform
        hop_length=160 # Number of samples between successive frames 10 ms
    )
    
    # 3. Convert to Decibel (dB) scale
    # Machine learning models perform much better on a logarithmic scale
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    # 4. Compute Deltas and Delta-Deltas
    delta = librosa.feature.delta(mel_spec_db)
    delta2 = librosa.feature.delta(mel_spec_db, order=2)
    
    # 5. Stack into 3 channels (Shape: 3, n_mels, Time)
    stacked_mel = np.stack([mel_spec_db, delta, delta2], axis=0)
    
    return stacked_mel.astype(np.float32)

In [38]:
print("Extracting Mel Spectrograms...")
train_df['mel_spec'] = train_df['signal_data'].apply(get_mel_spectrogram)
test_df['mel_spec'] = test_df['signal_data'].apply(get_mel_spectrogram)

print("Finished!")
print(f"Shape of first spectrogram: {test_df['mel_spec'].iloc[0].shape}")

Extracting Mel Spectrograms...
Finished!
Shape of first spectrogram: (3, 128, 47)


In [39]:
test_df['mel_spec'].iloc[5].shape

(3, 128, 41)

In [40]:
import numpy as np

# 1. Find the maximum time dimension (now axis 2) across BOTH datasets
train_max = train_df['mel_spec'].apply(lambda x: x.shape[2]).max()
test_max = test_df['mel_spec'].apply(lambda x: x.shape[2]).max()
global_max_len = max(train_max, test_max)

print(f"The dynamic maximum length across both sets is: {global_max_len} frames")
# Alternatively, force it to 160 as we discussed:
global_max_len = 160

# 2. Define the 3D padding function
def pad_3d_mel(stacked_mel, max_len):
    current_time = stacked_mel.shape[2]
    
    if current_time < max_len:
        pad_width = max_len - current_time
        
        # Split channels: 0 is Base Mel, 1 is Delta, 2 is Delta-Delta
        mel_chan = stacked_mel[0]
        delta_chan = stacked_mel[1]
        delta2_chan = stacked_mel[2]
        
        # Base Mel gets padded with its minimum value (absolute silence)
        mel_padded = np.pad(mel_chan, ((0, 0), (0, pad_width)), mode='constant', constant_values=mel_chan.min())
        # Deltas get padded with 0 (zero movement)
        delta_padded = np.pad(delta_chan, ((0, 0), (0, pad_width)), mode='constant', constant_values=0)
        delta2_padded = np.pad(delta2_chan, ((0, 0), (0, pad_width)), mode='constant', constant_values=0)
        
        return np.stack([mel_padded, delta_padded, delta2_padded], axis=0)
    
    # If it's already max_len (or longer, if you hardcoded 160), crop it
    return stacked_mel[:, :, :max_len]

# 3. Apply the padding
train_df['mel_spec_padded'] = train_df['mel_spec'].apply(lambda x: pad_3d_mel(x, global_max_len))
test_df['mel_spec_padded'] = test_df['mel_spec'].apply(lambda x: pad_3d_mel(x, global_max_len))

The dynamic maximum length across both sets is: 153 frames


In [41]:
def extract_dynamic_features_from_3d(stacked_mel):
    # stacked_mel shape is (3, 128, Time)
    # Channel 0: Mel, Channel 1: Delta, Channel 2: Delta-Delta
    
    # Take mean and variance across the time axis (axis 2)
    features = [
        np.mean(stacked_mel[0], axis=1), np.std(stacked_mel[0], axis=1),
        np.mean(stacked_mel[1], axis=1), np.std(stacked_mel[1], axis=1),
        np.mean(stacked_mel[2], axis=1), np.std(stacked_mel[2], axis=1)
    ]
    
    # Combine into a single flat vector (128 * 6 = 768 features)
    return np.concatenate(features)

# Apply to dataframe (make sure to use the UNPADDED 'mel_raw' so you don't calculate the mean of empty silence!)
train_df['mel_spec_pooled'] = train_df['mel_spec'].apply(extract_dynamic_features_from_3d)
test_df['mel_spec_pooled'] = test_df['mel_spec'].apply(extract_dynamic_features_from_3d)

In [42]:
print(train_df['mel_spec_padded'].iloc[0].shape)
print(train_df['mel_spec_pooled'].iloc[0].shape)
print(test_df['mel_spec_padded'].iloc[0].shape)
print(test_df['mel_spec_pooled'].iloc[0].shape)

(3, 128, 160)
(768,)
(3, 128, 160)
(768,)


In [43]:
test_df['mel_spec_padded'].iloc[0][2, 127, 159]

np.float32(0.0)

In [44]:
test_df.head()

,signal_data,language,speaker,label,gender,c,length,w2v_base_emb_pooled,w2v_base_emb,w2v_xlr_emb,w2v_xlr_emb_pooled,mel_spec,mel_spec_padded,mel_spec_pooled
104,"[-0.0234375, -0.015625, -0.0078125, -0.0078125...",SI,C,8,M,c,7500,"[0.172668, 0.16687207, 0.0005224917, 0.2749196...","[[0.13388804, 0.24878758, 0.16531222, 0.065995...","[[-0.26172817, -0.18423216, -0.0057340334, 0.1...","[-0.059850663, -0.13505645, 0.06951369, 0.0457...","[[[-39.23843, -37.075966, -39.140297, -39.8880...","[[[-39.23843, -37.075966, -39.140297, -39.8880...","[-39.28331, -40.02017, -45.294884, -36.950756,..."
218,"[-0.14410400390625, -0.144775390625, -0.145050...",SA,E,9,F,c,11251,"[0.12205984, 0.061295304, 0.0839988, 0.1493106...","[[0.122589916, 0.22048393, 0.010663711, -0.003...","[[-0.2639555, -0.18284531, -0.07761689, 0.1233...","[0.014569122, -0.10663602, 0.06223258, 0.02708...","[[[-2.942192, -0.72269726, -1.1388712, -0.7538...","[[[-2.942192, -0.72269726, -1.1388712, -0.7538...","[-1.129581, -4.0561967, -42.669647, -44.773624..."
64,"[-0.0078125, -0.0078125, -0.0078125, 0.0, -0.0...",IT,D,9,F,c,24456,"[0.13495167, 0.24337417, -0.120708704, 0.07979...","[[0.17708795, 0.2811785, -0.035754267, -0.0504...","[[-0.29650438, -0.13774817, -0.008006406, 0.12...","[-0.051639326, -0.078836694, 0.054401334, 0.05...","[[[-21.93634, -33.356495, -27.769451, -30.0775...","[[[-21.93634, -33.356495, -27.769451, -30.0775...","[-27.096249, -27.413214, -28.618872, -34.04058..."
6,"[-3.0517578125e-05, 0.0, 0.0, -3.0517578125e-0...",PO,A,7,M,c,10393,"[0.12713993, 0.2634376, -0.031140177, 0.104866...","[[0.17810075, 0.17139237, 0.16290793, -0.20194...","[[-0.38144502, -0.08141056, -0.061499953, 0.11...","[0.09064727, 0.025495477, 0.09566513, 0.012675...","[[[-80.0, -80.0, -48.818073, -52.50093, -31.01...","[[[-80.0, -80.0, -48.818073, -52.50093, -31.01...","[-24.92547, -26.25804, -32.901535, -37.486557,..."
127,"[-0.000152587890625, 0.0001220703125, 0.000152...",FR,B,4,F,c,7001,"[0.30866632, 0.3433385, -0.14996675, 0.3086379...","[[0.4475113, 0.18565086, 0.10488541, -0.090703...","[[-0.19600232, -0.08252267, 0.0015746434, 0.08...","[-0.0851352, -0.087434515, 0.05503741, 0.02675...","[[[-79.48943, -75.99508, -79.80349, -65.422646...","[[[-79.48943, -75.99508, -79.80349, -65.422646...","[-55.3433, -48.07939, -46.272297, -42.218052, ..."


In [45]:
# Save the training set
train_df.to_pickle("train_embedded.pkl")

# Save the testing set
test_df.to_pickle("test_embedded.pkl")